# FAERS Stage 1 consolidation — guided notebook

This notebook walks through [`build_stage1_consolidate.py`](./build_stage1_consolidate.py) one cell at a time. It explains how the script selected the latest case versions, removed deleted cases, retained matching child-table rows, and checked the result.

**Run cells from top to bottom with Shift+Enter.** Select the project `.venv` kernel first. This notebook reads the completed Stage 1 database and the eight source profile databases; it does **not** run the production consolidation script or modify any database.

The notebook uses the saved Stage 1 output in `outputs/stage1_consolidated_20261028`. If your output is in a different folder, change `STAGE1_DIR` in the setup cell.

## Before you begin: an important overwrite warning

The production script contains `if DB_PATH.exists(): DB_PATH.unlink()`. That deletes an existing output database before rebuilding it. Do not run that script against an output you want to keep unless you have a backup and have checked the configured output path. The notebook below is deliberately read-only.

The script's default `ROOT` and `OUT_DIR` values are placeholder paths. When run with two command-line arguments, those arguments replace the defaults. This notebook discovers the project folder and reads the already-created output instead.

In [ ]:
from pathlib import Path
import re
import duckdb
import pandas as pd
from IPython.display import display

# Find the repository even if Jupyter was opened from a subfolder.
here = Path.cwd().resolve()
PROJECT_DIR = next(
    (p for p in (here, *here.parents)
     if (p / "build_stage1_consolidate.py").is_file() and (p / ".git").exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from the project folder."
    )

OUTPUTS_DIR = PROJECT_DIR / "outputs"
STAGE1_DIR = OUTPUTS_DIR / "stage1_consolidated_20261028"
STAGE1_DB = STAGE1_DIR / "faers_stage1.duckdb"
STAGE1_REPORT = STAGE1_DIR / "stage1_report.txt"

if not STAGE1_DB.is_file():
    raise FileNotFoundError(f"Stage 1 database not found: {STAGE1_DB}")
if not STAGE1_REPORT.is_file():
    raise FileNotFoundError(f"Stage 1 report not found: {STAGE1_REPORT}")

print(f"Project: {PROJECT_DIR}")
print(f"Reading Stage 1 output: {STAGE1_DB}")
print(f"Report: {STAGE1_REPORT}")

## 1. Start with the run report

The script writes a text report as it works. The section headings, final row counts, and `[PASS]` / `[FAIL]` lines let us inspect what happened without rerunning the expensive consolidation. The final result should say `ALL CHECKS PASSED`.

In [ ]:
report_text = STAGE1_REPORT.read_text(encoding="utf-8")
important_lines = [
    line for line in report_text.splitlines()
    if line.startswith(("Run started:", "Quarters:", "Output database:",
                        "built stg_", "RESULT:", "Database size:", "Tables:", "Audit :"))
]
print("\n".join(important_lines))

## 2. Locate and attach the source databases — read-only

Each profile database contains a quarter's `demo`, `drug`, `reac`, `outc`, `rpsr`, `ther`, and `indi` tables, plus `del_ids`. The script discovers databases named `profile_??Q?.duckdb` under `ROOT`, attaches them with `READ_ONLY`, and checks that every required table exists. `LABELS` is the sorted list of quarter codes extracted from those filenames.

The following cell mirrors that input discovery and attaches both the source databases and the completed Stage 1 database to an in-memory DuckDB connection. The Stage 1 database is also attached read-only.

In [ ]:
TABLES = ["demo", "drug", "reac", "outc", "rpsr", "ther", "indi"]
found = {
    path.stem[-4:]: path
    for path in OUTPUTS_DIR.rglob("profile_??Q?.duckdb")
}
LABELS = sorted(found)
if len(LABELS) != 8:
    raise FileNotFoundError(
        f"Expected 8 quarter profile databases under {OUTPUTS_DIR}; found {LABELS}"
    )

con = duckdb.connect(":memory:")
con.execute("SET threads TO 4")
for quarter, path in found.items():
    safe_path = path.resolve().as_posix().replace("'", "''")
    con.execute(f"ATTACH '{safe_path}' AS db_{quarter} (READ_ONLY)")

safe_stage1_path = STAGE1_DB.resolve().as_posix().replace("'", "''")
con.execute(f"ATTACH '{safe_stage1_path}' AS stage1 (READ_ONLY)")

inventory = []
for quarter in LABELS:
    tables_here = {
        row[0] for row in con.execute(
            f"SELECT table_name FROM information_schema.tables "
            f"WHERE table_catalog = 'db_{quarter}'"
        ).fetchall()
    }
    missing = [name for name in TABLES + ["del_ids"] if name not in tables_here]
    if missing:
        raise ValueError(f"{quarter} is missing required tables: {missing}")
    demo_rows = con.execute(f"SELECT count(*) FROM db_{quarter}.demo").fetchone()[0]
    inventory.append({"quarter": quarter, "demo_rows": demo_rows,
                      "required_tables_present": True, "database": found[quarter].name})

display(pd.DataFrame(inventory))

## 3. Find the same sections in the Python script

This is a line-numbered map of the important blocks. Reading the source as text is safe: it does not execute the script. Notice that paths and output files are configured before the connection and that the database is unlinked if it already exists.

In [ ]:
script_lines = (PROJECT_DIR / "build_stage1_consolidate.py").read_text(
    encoding="utf-8"
).splitlines()

sections = [(1, 58), (59, 88), (89, 139), (140, 180), (181, 213)]
for start, end in sections:
    print(f"\n--- build_stage1_consolidate.py lines {start}-{end} ---")
    for number in range(start, min(end, len(script_lines)) + 1):
        print(f"{number:>3}: {script_lines[number - 1]}")

## 4. Build the combined case-version list (`_keys_all`)

For every quarter, the production script selects the case identifiers and the fields needed to compare versions: `primaryid`, `caseid`, `caseversion` (cast to an integer), and `fda_dt`. It adds `q_idx` (the quarter's sort position) and `source_quarter` (the quarter file supplying this version), then stacks all quarters together.

A row here is a **case appearance/version in one quarter**, not necessarily a unique case. The example below finds one case appearing in multiple quarters and shows its version history. It reads the already-built `_keys_all` table from the Stage 1 database.

In [ ]:
repeat_case = con.execute("""
    SELECT caseid
    FROM stage1._keys_all
    GROUP BY caseid
    HAVING count(*) > 1
    ORDER BY caseid
    LIMIT 1
""").fetchone()[0]

print(f"Example repeated caseid: {repeat_case}")
display(con.execute("""
    SELECT source_quarter, primaryid, caseid, ver AS caseversion, fda_dt, q_idx
    FROM stage1._keys_all
    WHERE caseid = ?
    ORDER BY q_idx
""", [repeat_case]).df())

## 5. Rank versions and choose one winner per case

The SQL `row_number()` window restarts for each `caseid`. The row with rank 1 is the winner. Ordering is by highest version, then latest `fda_dt`, then latest quarter index, then highest `primaryid`; `NULLS LAST` prevents a missing version/date from outranking a known one. In the actual data, later appearances had higher versions except for identical re-deliveries, but the tie-breaks make the rule explicit.

This cell applies the same ranking to just the example case and checks the chosen row against the saved `_winners` table.

In [ ]:
ranked_example = con.execute("""
    SELECT *, row_number() OVER (
        PARTITION BY caseid
        ORDER BY ver DESC NULLS LAST, fda_dt DESC NULLS LAST,
                 q_idx DESC, primaryid DESC
    ) AS rank_in_case
    FROM stage1._keys_all
    WHERE caseid = ?
    ORDER BY rank_in_case
""", [repeat_case]).df()
display(ranked_example)

saved_winner = con.execute("""
    SELECT source_quarter, primaryid, ver, fda_dt
    FROM stage1._winners WHERE caseid = ?
""", [repeat_case]).df()
print("Saved winner selected by the production run:")
display(saved_winner)

## 6. Exclude cases in the DELETE list

The script creates `_del` from the DELETE identifiers and filters those case IDs out of the winner list to make `_keep`. In this run, each of the eight profile databases has the same DELETE identifier set, so reading the list from the first quarter database is sufficient for these inputs. The comparison cell checks that fact rather than assuming it.

In [ ]:
delete_sets = {
    quarter: {row[0] for row in
              con.execute(f"SELECT caseid FROM db_{quarter}.del_ids").fetchall()}
    for quarter in LABELS
}
reference_set = delete_sets[LABELS[0]]
delete_comparison = [
    {"quarter": quarter,
     "distinct_delete_caseids": len(delete_sets[quarter]),
     "same_as_first_quarter": delete_sets[quarter] == reference_set,
     "ids_different_from_first": len(delete_sets[quarter] ^ reference_set)}
    for quarter in LABELS
]
display(pd.DataFrame(delete_comparison))

## 7. Understand the case funnel

The funnel separates three different counts: every case-version row across all quarterly files; one selected latest version per distinct case; and the remaining cases after DELETE-list exclusion. The saved audit table records the counts from the run, so we can inspect them without recomputing the selection.

In [ ]:
funnel = con.execute("""
    SELECT step, cases FROM stage1.audit_case_funnel
""").df()
display(funnel)

## 8. Keep child rows from the winning report version

`DRUG`, `REAC`, `OUTC`, `RPSR`, `THER`, and `INDI` are detail tables. The script keeps a child row only when its `primaryid` belongs to a kept case **and** the row came from the quarter that supplied that winning version. This prevents a case's earlier-version details from being mixed with its later-version details.

Each consolidated table gets one extra field, `source_quarter`. The script leaves the original columns and values unchanged; `source_quarter` records provenance, not the case's first-received quarter. `audit_table_rows` records source and retained row counts for each table and quarter.

In [ ]:
table_audit = con.execute("""
    SELECT table_name, source_quarter, rows_in_source, rows_kept, rows_removed
    FROM stage1.audit_table_rows
    ORDER BY table_name, source_quarter
""").df()
display(table_audit)

print("Totals across quarters:")
display(table_audit.groupby("table_name", as_index=False)[
    ["rows_in_source", "rows_kept", "rows_removed"]
].sum())

The next cell follows one actual kept case into `stg_demo` and `stg_drug`. It compares the quarter and report identifier, then counts that report's drug rows. The example is illustrative; the audit and integrity checks below cover all rows.

In [ ]:
example_case = con.execute("""
    SELECT caseid FROM stage1.stg_demo ORDER BY caseid LIMIT 1
""").fetchone()[0]
display(con.execute("""
    SELECT d.caseid, d.primaryid, d.source_quarter,
           count(g.primaryid) AS retained_drug_rows
    FROM stage1.stg_demo AS d
    LEFT JOIN stage1.stg_drug AS g
      ON d.primaryid = g.primaryid AND d.caseid = g.caseid
    WHERE d.caseid = ?
    GROUP BY d.caseid, d.primaryid, d.source_quarter
""", [example_case]).df())

## 9. What the audit tables and integrity checks prove

- `audit_case_funnel` records the selection counts.
- `audit_table_rows` records rows read, retained, and removed per table and quarter.
- `audit_case_lineage` shows each case's appearances, winning version, and deletion status.
- `audit_removed_deleted_cases` lists cases removed because they appeared in DELETE data.

The production run also checked one DEMO row per case, no deleted cases in output, no orphan child rows, matching child/DEMO quarter and case IDs, text-only output columns, and row-by-row source fidelity. These full-table checks were already run by the script; this notebook displays their saved results instead of repeating expensive scans.

In [ ]:
check_lines = [
    line for line in report_text.splitlines()
    if line.startswith(("[PASS]", "[FAIL]", "RESULT:"))
]
print("\n".join(check_lines))

if any(line.startswith("[FAIL]") for line in check_lines):
    raise RuntimeError("The saved Stage 1 report contains a failed integrity check.")
if not any("ALL CHECKS PASSED" in line for line in check_lines):
    raise RuntimeError("The report does not confirm that all Stage 1 checks passed.")

## 10. Inspect the saved audit detail and consolidated tables

The selection tables are retained in the output database to make the decisions inspectable. The queries below display only a few rows. The `stg_*` tables are consolidated **staging tables**, not cleaned or visualization-ready fact tables.

In [ ]:
print("A few case-lineage rows:")
display(con.execute("""
    SELECT caseid, n_appearances, first_seen_q, last_seen_q, max_version,
           winning_quarter, winning_primaryid, first_delete_q, was_deleted
    FROM stage1.audit_case_lineage
    ORDER BY n_appearances DESC, caseid
    LIMIT 8
""").df())

print("A few retained DEMO rows (original values plus source_quarter):")
display(con.execute("""
    SELECT caseid, primaryid, caseversion, init_fda_dt, fda_dt,
           age, age_cod, age_grp, sex, source_quarter
    FROM stage1.stg_demo
    ORDER BY caseid
    LIMIT 8
""").df())

## 11. What Stage 1 intentionally does not do

Stage 1 preserves delivered fields as text and adds only `source_quarter`. It does **not** standardize ages, dates, sex, occupation, outcomes, drug names, or ingredients; deduplicate reaction rows; infer a missing primary suspect; remove `DN` rows; or produce chart-ready aggregates.

Those are Stage 2 decisions. In particular, keep the raw values alongside cleaned values and quality flags, convert age using `AGE_COD`, preserve partial-date precision, don't treat non-administered drugs as exposures, and keep drug ingredient roll-ups reviewable. FAERS reports describe suspected adverse events and do not establish that a drug caused an event.

## 12. Close the notebook connection

This closes only the notebook's in-memory DuckDB connection. All source profile databases and the Stage 1 database were attached read-only.

In [ ]:
con.close()
print("Notebook connection closed. No source or Stage 1 database was modified.")